# 01 — GLORYS12 monthly subset over the `egsl` envelope

Downloads monthly-mean surface ocean and sea-ice fields from the Copernicus Marine reanalysis
`GLOBAL_MULTIYEAR_PHY_001_030` (GLORYS12V1) over the lon/lat bounds of ice-clim's `egsl` mask.

In [2]:
from pathlib import Path

import copernicusmarine
import geopandas as gpd

EGSL_MASK_PATH = Path.home() / "data/masks/climatology_bbox/egsl/egsl_32198_bbox.geojson"
OUT_DIR = Path.home() / "data/CMS/GLORYS"

MASK_CRS = "EPSG:32198"  # Québec Lambert, metres
GEOGRAPHIC_CRS = "EPSG:4326"
DENSIFY_MAX_SEGMENT_M = 1_000  # straight Lambert edges curve in lon/lat; 1 km vertices track the curve

/home/eliedl/Documents/profesionnel/glorys-lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Mask → lon/lat bounds

Densify the edges in the projected CRS before reprojecting: reprojecting only the corners would
miss the bulge of the edges in lon/lat and understate the bounds.

In [3]:
egsl_mask = gpd.read_file(EGSL_MASK_PATH)
assert egsl_mask.crs == MASK_CRS, egsl_mask.crs

egsl_4326 = egsl_mask.geometry.segmentize(DENSIFY_MAX_SEGMENT_M).to_crs(GEOGRAPHIC_CRS)
min_lon, min_lat, max_lon, max_lat = egsl_4326.total_bounds
print(f"lon {min_lon:.4f} → {max_lon:.4f}   lat {min_lat:.4f} → {max_lat:.4f}")

lon -71.9548 → -53.2824   lat 42.0272 → 52.5891


## Request spec

In [4]:
DATASET_ID = "cmems_mod_glo_phy_my_0.083deg_P1M-m"
DATASET_VERSION = "202311"
VARIABLES = [
    "siconc", "sithick", "usi", "vsi",  # sea ice
    "uo", "vo", "thetao", "so", "zos",  # ocean surface
]
SURFACE_MAX_DEPTH_M = 0.5  # selects the top model level only (0.494 m)
START = "1993-09-01"
END = "2022-09-01"
OUT_FILENAME = "glorys12_P1M_egsl_19930901-20220901.nc"

## Login

Run once per machine: stores credentials in `~/.copernicusmarine/`, never in this notebook.

In [4]:
copernicusmarine.login()

INFO - 2026-11-11T15:20:29Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register
INFO - 2026-11-11T15:24:28Z - Credentials file stored in /home/eliedl/.copernicusmarine/.copernicusmarine-credentials.


True

## Subset

`coordinates_selection_method="inside"`: keeps grid cells whose centres fall within the bounds (no buffer).
Run with `DRY_RUN = True` first to inspect the response (grid, size) before downloading.

In [ ]:
DRY_RUN = False

response = copernicusmarine.subset(
    dataset_id=DATASET_ID,
    dataset_version=DATASET_VERSION,
    variables=VARIABLES,
    minimum_longitude=min_lon,
    maximum_longitude=max_lon,
    minimum_latitude=min_lat,
    maximum_latitude=max_lat,
    minimum_depth=0,
    maximum_depth=SURFACE_MAX_DEPTH_M,
    start_datetime=START,
    end_datetime=END,
    coordinates_selection_method="inside",
    output_directory=OUT_DIR,
    output_filename=OUT_FILENAME,
    dry_run=DRY_RUN,
)
response

INFO - 2026-11-11T15:29:49Z - Selected dataset version: "202311"
INFO - 2026-11-11T15:29:49Z - Selected dataset part: "default"
WARNING - 2026-11-11T15:29:49Z - Some of your subset selection [0.0, 0.5] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]
100%|██████████| [00:06<00:00]
INFO - 2026-11-11T15:29:57Z - Total size of the download: 170.54 MB.


ResponseSubset(file_path=PosixPath('/home/eliedl/data/CMS/GLORYS/glorys12_P1M_egsl_19930901-20220901.nc'), output_directory=PosixPath('/home/eliedl/data/CMS/GLORYS'), filename='glorys12_P1M_egsl_19930901-20220901.nc', file_size=170.54348091603055, data_transfer_size=1990.0931908396942, variables=['siconc', 'sithick', 'usi', 'vsi', 'uo', 'vo', 'thetao', 'so', 'zos'], coordinates_extent=[GeographicalExtent(minimum=-71.91666412353516, maximum=-53.33333206176758, unit='degrees_east', coordinate_id='longitude'), GeographicalExtent(minimum=42.08333206176758, maximum=52.58333206176758, unit='degrees_north', coordinate_id='latitude'), TimeExtent(minimum='1993-09-01T00:00:00+00:00', maximum='2022-09-01T00:00:00+00:00', unit='iso8601', coordinate_id='time'), GeographicalExtent(minimum=0.49402499198913574, maximum=0.49402499198913574, unit='m', coordinate_id='depth')], status='000', message='The request was successful.', file_status='DOWNLOADED', file_names=None)

## Daily `siconc` subset

Same bounds, period and surface-depth selection as the monthly request, from the daily-mean dataset.
`siconc` is a 2-D field, so the depth bounds have no effect on it; they are kept for parity with the monthly request.

In [6]:
DAILY_DATASET_ID = "cmems_mod_glo_phy_my_0.083deg_P1D-m"
DAILY_VARIABLES = ["siconc"]
DAILY_OUT_FILENAME = "glorys12_P1D_siconc_egsl_19930901-20220901.nc"
DRY_RUN = False

daily_response = copernicusmarine.subset(
    dataset_id=DAILY_DATASET_ID,
    dataset_version=DATASET_VERSION,
    variables=DAILY_VARIABLES,
    minimum_longitude=min_lon,
    maximum_longitude=max_lon,
    minimum_latitude=min_lat,
    maximum_latitude=max_lat,
    minimum_depth=0,
    maximum_depth=SURFACE_MAX_DEPTH_M,
    start_datetime=START,
    end_datetime=END,
    coordinates_selection_method="inside",
    output_directory=OUT_DIR,
    output_filename=DAILY_OUT_FILENAME,
    dry_run=DRY_RUN,
)
daily_response

INFO - 2026-11-11T16:32:04Z - Selected dataset version: "202311"
INFO - 2026-11-11T16:32:04Z - Selected dataset part: "default"
WARNING - 2026-11-11T16:32:04Z - Some of your subset selection [0.0, 0.5] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]
100%|██████████| [00:09<00:00]
INFO - 2026-11-11T16:32:15Z - Total size of the download: 575.19 MB.


ResponseSubset(file_path=PosixPath('/home/eliedl/data/CMS/GLORYS/glorys12_P1D_siconc_egsl_19930901-20220901.nc'), output_directory=PosixPath('/home/eliedl/data/CMS/GLORYS'), filename='glorys12_P1D_siconc_egsl_19930901-20220901.nc', file_size=575.1913206106871, data_transfer_size=3294.0164885496183, variables=['siconc'], coordinates_extent=[GeographicalExtent(minimum=-71.91666412353516, maximum=-53.33333206176758, unit='degrees_east', coordinate_id='longitude'), GeographicalExtent(minimum=42.08333206176758, maximum=52.58333206176758, unit='degrees_north', coordinate_id='latitude'), TimeExtent(minimum='1993-09-01T00:00:00+00:00', maximum='2022-09-01T00:00:00+00:00', unit='iso8601', coordinate_id='time')], status='000', message='The request was successful.', file_status='DOWNLOADED', file_names=None)